In [7]:
import os
import numpy as np
import pandas as pd
import mido
from mido import MidiFile, MidiTrack, MetaMessage, Message
from tqdm.notebook import tqdm

data_folder = "data"
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)
sec_base = True
sec_unit = 0.1

###################################################
############      MIDI Information
###################################################

def print_midi_info(mid):
    print("File Name: ", mid.filename)
    print("Total Length: ", mid.length)
    
    if mid.tracks:
        total_tracks = len(mid.tracks)
        print("-" * 70)
        print("Track Name: ", mid.tracks[0].name)
        print("Number of Tracks: ", total_tracks)
        
        for i in range(total_tracks):
            track_name = mid.tracks[i].name if mid.tracks[i].name else "No named"
            print(f"[{i+1}]. {track_name}")
    else:
        print("There are no track.")

def load_midi_data(input_name):
    input_path = os.path.join(data_folder, input_name)
    input_mid = mido.MidiFile(input_path)
    
    print("=" * 20, "[Performance Midi Data]", "=" * 20)
    print_midi_info(input_mid)
    
    return input_mid

def create_df_info():
    df_col = ['sec', 'msg_type', 'channel', 'note', 'velocity', 'dynamic', 'accent', 'count',
              'main_vol', 'depth', 'pedal', 'pan']
    df = pd.DataFrame(columns=df_col)
    
    df_tempo_col = ['tempo', 'tick']
    df_tempo = pd.DataFrame(columns=df_tempo_col)

    df_info = {MetaMessage: {}, Message: df}
    
    return df_info, df_tempo

def info_to_list(cur_info):
    return [cur_info['sec'], cur_info['msg_type'], cur_info['channel'], cur_info['note'],
            cur_info['velocity'], cur_info['dynamic'], cur_info['accent'], cur_info['count'],
            cur_info['main_vol'], cur_info['depth'], cur_info['pedal'], cur_info['pan']]

def initialize_cur_info():
    return {'sec': 0, 'msg_type': [], 'channel': [], 'note': [], 'velocity': [], 'dynamic': "", 'accent': 0, 'count': 0,
            'main_vol': -1, 'depth': -1, 'pedal': -1, 'pan': -1}

def check_not_list(msg_list):
    return msg_list if isinstance(msg_list, list) else [msg_list]

def tempo_info_list(mid_info, tempo_info):
    if 'set_tempo' not in mid_info[MetaMessage]:
        msg = mido.MetaMessage('set_tempo', tempo=500000)
        mid_info[MetaMessage][msg.type] = [msg]
    
    tempo_list = mid_info[MetaMessage]['set_tempo']
    tempo_tick = 0
    
    for idx, msg in enumerate(tempo_list):
        tempo_tick += msg.time
        tempo_info.loc[idx, 'tempo'] = msg.tempo
        if idx > 0:
            tempo_info.loc[idx - 1, 'tick'] = tempo_tick
            
    tempo_info.loc[tempo_info.shape[0] - 1, 'tick'] = -1
    return tempo_info

def find_tempo(tick, tempo_info):
    last_tempo_idx = tempo_info.shape[0]
    for tempo_idx in range(last_tempo_idx):
        if tempo_info.loc[tempo_idx, 'tick'] > tick or tempo_info.loc[tempo_idx, 'tick'] == -1:
            return tempo_info.loc[tempo_idx, 'tempo']
    return tempo_info.loc[last_tempo_idx - 1, 'tempo']

###################################################
############      Dynamic
###################################################
def distribute_dynamic(mean_velocity):
    if 0 <= mean_velocity < 36: return 'ppp'
    if 36 <= mean_velocity < 49: return 'pp'
    if 49 <= mean_velocity < 62: return 'p'
    if 62 <= mean_velocity < 75: return 'mp'
    if 75 <= mean_velocity < 88: return 'mf'
    if 88 <= mean_velocity < 101: return 'f'
    if 101 <= mean_velocity < 114: return 'ff'
    if 114 <= mean_velocity <= 127: return 'fff'
    return ''

###################################################
############  Dynamic and accent
###################################################
def calculate_dynamic_accent(cur_info, msg_df):
    last_idx = msg_df.shape[0]
    prev_avg_velo = 0
    if last_idx > 0:
        prev_velo = msg_df.loc[last_idx - 1, 'velocity']
        if prev_velo:
            prev_avg_velo = sum(prev_velo) / len(prev_velo)
            
    mean_velocity = 0
    if cur_info['count'] > 0:
        mean_velocity = sum(cur_info['velocity']) / len(cur_info['velocity'])
        
    dynamic = distribute_dynamic(mean_velocity)
    accent = 1 if 76 <= mean_velocity <= 127 and mean_velocity > 1.2 * prev_avg_velo else 0
    
    return dynamic, accent

def control_type(msg):
    if msg.control == 7: return 'main_vol'
    if msg.control == 10: return 'pan'
    if msg.control == 64: return 'pedal'
    if 91 <= msg.control <= 93: return 'depth'
    return None

def process_msg(msg, info):
    if msg.type in ('note_on', 'note_off'):
        msg_type = 'note_off' if msg.type == 'note_on' and msg.velocity == 0 else msg.type
        info['msg_type'].append(msg_type)
        info['channel'].append(msg.channel)
        info['note'].append(msg.note)
        info['velocity'].append(msg.velocity)
        info['count'] += 1
    elif msg.type == 'control_change':
        ctl_type = control_type(msg)
        if ctl_type:
            if info[ctl_type] == -1:
                info[ctl_type] = msg.value
            else:
                info[ctl_type] = check_not_list(info[ctl_type])
                info[ctl_type].append(msg.value)
    return True

def remain_check(info):
    return (info['count'] > 0 or info['main_vol'] != -1 or info['depth'] != -1 or
            info['pedal'] != -1 or info['pan'] != -1)

def extract_mid_data(mid):
    mid_info, tempo_info = create_df_info()
    
    for idx, track in enumerate(mid.tracks):
        if idx == 0:
            for msg in track:
                if isinstance(msg, MetaMessage) and msg.type not in mid_info[MetaMessage]:
                    mid_info[MetaMessage][msg.type] = []
                mid_info[MetaMessage][msg.type].append(msg)
            tempo_info_list(mid_info, tempo_info)
            continue

        cur_tick = 0
        cur_sec = 0
        cur_info = initialize_cur_info()

        for msg in tqdm(track, desc=f"Track {idx+1}"):
            if isinstance(msg, Message):
                cur_tick += msg.time
                msg_tempo = find_tempo(cur_tick, tempo_info)
                msg_sec = mido.tick2second(cur_tick, mid.ticks_per_beat, msg_tempo)
                msg_sec = np.trunc(msg_sec * 10) / 10

                if msg_sec > cur_sec:
                    dynamic, accent = calculate_dynamic_accent(cur_info, mid_info[Message])
                    cur_info['dynamic'] = dynamic
                    cur_info['accent'] = accent
                    cur_info['sec'] = cur_sec
                    
                    info_last = mid_info[Message].shape[0]
                    mid_info[Message].loc[info_last] = info_to_list(cur_info)
                    
                    if sec_base:
                        temp_info = initialize_cur_info()
                        for sec in range(int(cur_sec * 10) + 1, int(msg_sec * 10)):
                            temp_info['sec'] = float(sec / 10)
                            mid_info[Message].loc[mid_info[Message].shape[0]] = info_to_list(temp_info)
                    
                    cur_info = initialize_cur_info()
                
                process_msg(msg, cur_info)
                cur_sec = msg_sec
        
        if remain_check(cur_info):
            dynamic, accent = calculate_dynamic_accent(cur_info, mid_info[Message])
            cur_info['dynamic'] = dynamic
            cur_info['accent'] = accent
            cur_info['sec'] = cur_sec
            mid_info[Message].loc[mid_info[Message].shape[0]] = info_to_list(cur_info)
            
    mid_info[Message].replace(-1, 0, inplace=True)
    
    base_filename = mid.filename.split("/")[-1].split(".")[0]
    csv_name = os.path.join(output_folder, f"{base_filename}_features.csv")
    mid_info[Message].to_csv(csv_name, index=False)
    print(f"\n'{csv_name}' Saved.")
    
    return mid_info

In [8]:
# The MIDI file name
input_name = "beethoven_dynamic_perfect1.midi"

input_mid = load_midi_data(input_name)

print("\n===== Extracting the features from MIDI =====")
input_info = extract_mid_data(input_mid)

print("\n" + "=" * 20, "  [Extracted Features]  ", "=" * 20)
display(input_info[Message])

==================== [Performance Midi Data] ====================
File Name:  data/beethoven_dynamic_perfect1.midi
Total Length:  101.34624999999961
----------------------------------------------------------------------
Track Name:  Sitar
Number of Tracks:  2
[1]. Sitar
[2]. dynamic_perfect1

===== Extracting the features from MIDI =====


Track 2:   0%|          | 0/3022 [00:00<?, ?it/s]


'output/beethoven_dynamic_perfect1_features.csv' Saved.

====================   [Extracted Features]   ====================


,sec,msg_type,channel,note,velocity,dynamic,accent,count,main_vol,depth,pedal,pan
0,0.0,[],[],[],[],ppp,0,0,127,0,0,64
1,0.1,[],[],[],[],,0,0,0,0,0,0
2,0.2,[],[],[],[],,0,0,0,0,0,0
3,0.3,[],[],[],[],,0,0,0,0,0,0
4,0.4,[],[],[],[],,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1009,100.9,[],[],[],[],,0,0,0,0,0,0
1010,101.0,[],[],[],[],,0,0,0,0,0,0
1011,101.1,"[note_on, note_off, note_on]","[0, 0, 0]","[76, 76, 88]","[75, 0, 82]",p,0,3,0,0,"[126, 120, 114, 110, 108]",0
1012,101.2,"[note_on, note_off]","[0, 0]","[76, 88]","[59, 0]",ppp,0,2,0,0,"[106, 104, 102, 94, 82]",0
